In [ ]:
import math
import torch
from torch import nn

# 目的是将所有不在validlen里的都给mask掉
def sequence_mask(X, valid_len, value = 0):
    maxlen = X.size(1)
    # 取X中的key的维度 因为要据此与validlen比较 同时将validlen给repeatinterleave
    # 这一步用到了broadcasting相当于前一个与后一个对齐 同时后一个也要broadcast来多次比较
    mask = torch.arange(maxlen,device = X.device)[None,:] < valid_len[: ,None]
    X = X.clone()
    # 相当于将false取反得到true的地方的值替换成value
    X[~mask] = value
    return X

def masked_softmax(X, valid_lens):
    if valid_lens is None:
        return nn.functional.softmax(X, dim = -1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            # 把validlens里的元素每个重复shape【1】次，便于后续对齐
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)

        X = sequence_mask(X.reshape(-1,shape[-1]),valid_lens, value =-1e6)
        # 沿着最后一行的键值对来softmask
        return nn.functional.softmax(X.reshape(shape),dim = -1)



加性注意力

$$a(\mathbf q, \mathbf k) = \mathbf w_v^\top \text{tanh}(\mathbf W_q\mathbf q + \mathbf W_k \mathbf k) \in \mathbb{R},$$

In [ ]:
from typing import Any


class AdditiveAttention(nn.Module):
    def __init__(self, key_size, query_size, num_hiddens, dropout, **kwargs: Any) -> None:
        super(AdditiveAttention, self).__init__( **kwargs)
        self.W_k = nn.Linear(key_size,num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias = False)
        self.w_v = nn.Linear(num_hiddens, 1, bias = False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        # 该线性变换的意义在于将query和key投影到同一个hiddenlayer
        queries, keys = self.W_q(queries), self.W_k(keys)
        # 各插一个维度后利用广播生成qk对的矩阵
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        scores = self.w_v(features).squeeze(-1)
        # 删去最后一个无意义维度后,scores的形状：(batch_size，查询的个数，“键-值”对的个数)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights),values)

queries, keys = torch.normal(0, 1, (2, 1, 20)), torch.ones((2,10,2))


上述tanh天然有解 不需缩放
缩放点积 attention 的提出目的:为了让不同维度下的点积数值稳定(纬度越高可能的点积值越大) 避免 softmax 过于尖锐，从而保证训练稳定和表达能力

In [ ]:
class DotProductAttention(nn.Module):
    def __init__(self, dropout, **kwargs: Any) -> None:
        super().__init__(dropout, **kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens = None):
        d = queries.shape[-1]
        scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
        self.attention_weights = masked_softmax(scores,valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)

query	现在要找什么    
key	    每个候选项的“标签/索引”      
value	每个候选项的“内容” 